# Skybox Generator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joshmcculloch/skybox-generator/blob/main/skybox_colab.ipynb)

Render a textured, terrain-accurate **equirectangular skybox** for any location
on Earth — satellite imagery for colour + SRTM elevation for relief.

**How to use:**
1. Run the install cell below (once per session).
2. Adjust the controls in the form, or use the **▶** button to re-run.
3. The skybox image appears below the form; right-click it to save.

> No API keys needed. Everything runs in your browser session.


In [ ]:
#@title Install skybox-generator
import os, subprocess, sys

REPO = "https://github.com/joshmcculloch/skybox-generator"
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"git+{REPO}"])

# Cache tiles in the session (resets each time; mount Drive + point here to persist)
os.environ.setdefault("SKYBOX_CACHE", os.path.join(os.getcwd(), "skybox_cache"))

import skybox
print(f"skybox-generator {skybox.__version__} ready")


## Example locations

| Place | `--lat` | `--lon` | Suggested `eye_height` |
|---|---|---|---|
| Matterhorn | `46.0216` | `7.7480` | `1000` |
| Grand Canyon | `36.0986` | `-112.1160` | `400` |
| London | `51.5074` | `-0.1278` | `2000` |
| Mount Fuji | `35.3606` | `138.7274` | `1200` |
| Pisa | `43.7228` | `10.3966` | `60` |

> Tip: SRTM coverage ends at ±60° latitude — near the poles terrain is flat.
> For a scenic mountain panorama raise `eye_height` above the valley floor.


In [ ]:
#@title Render a skybox { run: "auto", display-mode: "both" }
#@markdown Adjust the controls below — the skybox re-renders automatically.

lat = 46.0216           #@param {type:"number"}
lon = 7.7480            #@param {type:"number"}
sky = "clear"           #@param ["clear", "cloudy", "evening"]
width = 1024            #@param {type:"slider", min:256, max:2048, step:256}
radius_km = 60          #@param {type:"slider", min:10, max:120, step:5}
terrain_res = 512       #@param {type:"slider", min:256, max:1024, step:128}
eye_height = 1000       #@param {type:"number"}
heading = 0             #@param {type:"slider", min:0, max:360, step:15}
lod_count = 8           #@param {type:"slider", min:1, max:10, step:1}
custom_sun = False      #@param {type:"boolean"}
sun_azimuth = 225       #@param {type:"number"}
sun_elevation = 40      #@param {type:"number"}
n_steps = 192           #@param {type:"slider", min:48, max:384, step:16}
ambient = 0.4           #@param {type:"slider", min:0, max:1, step:0.1}
seed = 1                #@param {type:"integer"}

from IPython.display import Image, display
from skybox import generate

out = "skybox.png"
generate(
    lat=lat, lon=lon, sky_name=sky, out=out,
    width=int(width), radius_km=float(radius_km), terrain_res=int(terrain_res),
    eye_height=float(eye_height), heading_deg=float(heading),
    lod_count=int(lod_count),
    sun_azimuth=sun_azimuth if custom_sun else None,
    sun_elevation=sun_elevation if custom_sun else None,
    n_steps=int(n_steps), ambient=float(ambient), seed=int(seed),
)
display(Image(out))


## Notes

- First render is slower because SRTM/imagery tiles are downloaded (~30 m elevation,
  Esri World Imagery); later runs reuse the session cache.
- Terrain is a finite box (`radius_km`); beyond it distance haze fades into the sky.
- Higher `terrain_res` and `lod_count` give crisper relief but take longer.
- Output is a 2:1 equirectangular PNG — drop it straight into Blender / Unity / Unreal.
